In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
# import constants for the days of the week
from matplotlib.dates import MO, TU, WE, TH, FR, SA, SU
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec

from datetime import timedelta, datetime

# Importing everything

In [ ]:
rme_export = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_export.csv', index_col='Datetime', parse_dates=True)
rme_discharge = pd.read_excel('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rme_discharge.xlsx', index_col='Datetime', parse_dates=True)
rmsp3_combined = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rmsp3combined.csv', index_col='Datetime', parse_dates=True)
combined_176 = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/176combined.csv',index_col='Datetime', parse_dates=True)
soilmoisture_mbsec = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/soilmoisture_mbsec.csv',index_col='Datetime', parse_dates=True)
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/'
rme_results = pd.read_csv(result_dir+'rme_aa500.csv', index_col='Sample Datetime', keep_default_na=False, na_values='NaN', parse_dates=True)
rme_results = rme_results.drop(['2/15/25 3:30:00', '2-22-25 11:00']) # drop outliers
# Filter out VOL flagged results
rme_results_novol = rme_results[~rme_results['Nitrate QA'].str.contains('VOL')]
rme_results_vol = rme_results[rme_results['Nitrate QA'].str.contains('VOL')]
combined_176

In [ ]:
rme_export_hourly = rme_export.drop(columns='Date').resample('1h').mean()

In [ ]:
rme_discharge

# Export Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharex=False)
ax = np.array(ax)  # Ensure ax is a numpy array for indexing

start_date = '1/1/2025'
end_date = '12/31/2025'

second_y = ax[0].twinx()
second_y.invert_yaxis()
rme_discharge.loc[start_date:end_date].plot(y='qls', logy=False, title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
ax[0].set_ylim([.0005, 100])
second_y.set_ylim([10, 0])

# Combine legends from both axes
handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles = handles1 + handles2
labels = labels1 + labels2

# Add the combined legend
ax[0].legend(handles, labels, loc='upper left')
ax[0].set_ylabel('Discharge (L/s)')
second_y.set_ylabel('Hourly Precipitation (mm)')

#combined_176.loc[start_date:end_date].plot(y='tmp3', ax=ax[1], title='Air Temp', color='tab:red', x_compat=True, label='Air Temperature')
#ax[1].axhline(0, color='tab:gray', linestyle='--', label='Freezing')
#ax[1].set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
#ax[1].legend(loc='upper left')

rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[1], title = 'Nitrogen', color='tab:green',x_compat=True, label = 'NO3-N')
#rme_export.loc[start_date:end_date].plot(y='Ammonium mean', ax=ax[1], title = 'Nitrogen',x_compat=True, label = 'NH4-N', ylabel='Nitrogen Concentration (mg/L as N)')

ax[2].set_ylabel('Nitrogen (mg/L)')

rme_results_novol[start_date:end_date].plot(y='Nitrate mean', yerr = 'Nitrate err', marker = '*', color = 'tab:green', label= 'Sample NO3-N', ax=ax[1],x_compat=True, linestyle='None')
rme_results_novol[start_date:end_date].plot(y='Ammonium mean', yerr = 'Ammonium err', marker = '*', color = 'tab:blue', label= 'Sample NH4-N', ax=ax[1], x_compat=True)

rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct Export', ax=ax[2], color = 'tab:green',title = 'Nitrate Export', x_compat=True, ylabel = 'Nitrogen Export (kg/ha)', label='15min NO3-N Export (kg/ha)', legend=False)
rme_export.loc[start_date:end_date].plot(y='Ammonium mean Export', ax=ax[2], x_compat=True, label='15min NH4-N Export (kg/ha)')

secondary_y3 = ax[2].twinx()  # Create a secondary y-axis
rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct Cumulative Export', color='tab:green',ax=secondary_y3, x_compat=True, ylabel='Cumulative N Export (kg/ha)', label = 'Cumulative NO3-N Export', legend=False,linestyle='--')
rme_export.loc[start_date:end_date].plot(y='Ammonium mean Cumulative Export',ax=secondary_y3, x_compat=True, label = 'Cumulative NH4-N Export', color='tab:blue', legend=False,linestyle='--')
rme_export.loc[start_date:end_date].plot(y='TIN Cumulative Export',ax=secondary_y3, x_compat=True, label = 'Cumulative DIN Export', color='tab:cyan', legend=False,linestyle='--')



# Combine legends from both axes
handles1_3, labels1_3 = ax[2].get_legend_handles_labels()  # Primary y-axis
handles2_3, labels2_3 = secondary_y3.get_legend_handles_labels()  # Secondary y-axis
handles_3 = handles1_3 + handles2_3
labels_3 = labels1_3 + labels2_3
ax[2].legend(handles_3, labels_3, loc='upper left')


for a in ax:
    a.set_xlabel(None)
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

for a in ax[list(range(0,2))]:
    a.set_xticklabels([])


fig.tight_layout()





#plt.xticks(rotation=45)


#plt.tight_layout()
#plt.show()

#ax[3].get_xticks()



In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharex=False)
    ax = np.array(ax)  # Ensure ax is a numpy array for indexing

    start_date = '1/1/2025'
    end_date = '12/31/2025'

    second_y = ax[0].twinx()
    second_y.invert_yaxis()
    rme_discharge.loc[start_date:end_date].plot(y='qls', logy=False, title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    ax[0].set_ylim([.0005, 100])
    second_y.set_ylim([10, 0])

    # Combine legends from both axes
    handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    # Add the combined legend
    ax[0].legend(handles, labels, loc='lower right')
    ax[0].set_ylabel('Discharge (L/s)')
    second_y.set_ylabel('Hourly Precip (mm)')

    #combined_176.loc[start_date:end_date].plot(y='tmp3', ax=ax[1], title='Air Temp', color='tab:red', x_compat=True, label='Air Temperature')
    #ax[1].axhline(0, color='tab:gray', linestyle='--', label='Freezing')
    #ax[1].set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
    #ax[1].legend(loc='upper left')

    rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct', ax=ax[1], title = 'Nitrate', color='tab:green',x_compat=True, label = 'NO3-N')
    #rme_export.loc[start_date:end_date].plot(y='Ammonium mean', ax=ax[1], title = 'Nitrogen',x_compat=True, label = 'NH4-N', ylabel='Nitrogen Concentration (mg/L as N)')

    ax[1].set_ylabel('Nitrogen (mg/L)')

    rme_results_novol[start_date:end_date].plot(y='Nitrate mean', yerr = 'Nitrate err', marker = '*', color = 'k', label= 'Sample NO3-N', ax=ax[1],x_compat=True, linestyle='None')
    #rme_results_novol[start_date:end_date].plot(y='Ammonium mean', yerr = 'Ammonium err', marker = '*', color = 'tab:blue', label= 'Sample NH4-N', ax=ax[1], x_compat=True)

    rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct Export', ax=ax[2], color = 'tab:green',title = 'Nitrate Export', x_compat=True, ylabel = 'Export (kg/ha)', label='15min NO3-N Export (kg/ha)', legend=False)
    #rme_export.loc[start_date:end_date].plot(y='Ammonium mean Export', ax=ax[2], x_compat=True, label='15min NH4-N Export (kg/ha)')

    secondary_y3 = ax[2].twinx()  # Create a secondary y-axis
    rme_export.loc[start_date:end_date].plot(y='second_derivative_no3_mgl_bias_correct Cumulative Export', color='k',ax=secondary_y3, x_compat=True, ylabel='Cumulative Export (kg/ha)', label = 'Cumulative NO3-N Export', legend=False,linestyle='--')
    #rme_export.loc[start_date:end_date].plot(y='Ammonium mean Cumulative Export',ax=secondary_y3, x_compat=True, label = 'Cumulative NH4-N Export', color='tab:blue', legend=False,linestyle='--')
    #rme_export.loc[start_date:end_date].plot(y='TIN Cumulative Export',ax=secondary_y3, x_compat=True, label = 'Cumulative DIN Export', color='tab:cyan', legend=False,linestyle='--')



    # Combine legends from both axes
    handles1_3, labels1_3 = ax[2].get_legend_handles_labels()  # Primary y-axis
    handles2_3, labels2_3 = secondary_y3.get_legend_handles_labels()  # Secondary y-axis
    handles_3 = handles1_3 + handles2_3
    labels_3 = labels1_3 + labels2_3
    ax[2].legend(handles_3, labels_3, loc='lower right')


    for a in ax:
        a.set_xlabel(None)
        a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
        a.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    for a in ax[list(range(0,2))]:
        a.set_xticklabels([])

    ax[1].set_yticks([0, .2, .4, .6])

    ax[2].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

    ax[2].set_xticklabels(ax[2].get_xticklabels(), rotation=0, ha='center')

    fig.tight_layout()




# Season Overview Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5), nrows=3, sharex=False)
ax = np.array(ax)  # Ensure ax is a numpy array for indexing

start_date = '01/24/2025'
end_date = '06/01/2025'

second_y = ax[0].twinx()
second_y.invert_yaxis()
rme_discharge.loc[start_date:end_date].plot(y='qls', title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
ax[0].set_ylim([.0005, 100])
second_y.set_ylim([10, 0])

# Combine legends from both axes
handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles = handles1 + handles2
labels = labels1 + labels2

# Add the combined legend
ax[0].legend(handles, labels, loc='upper left')
ax[0].set_ylabel('Discharge (L/s)')
second_y.set_ylabel('Hourly Precipitation (mm)')

#secondary_y3 = ax[1].twinx()  # Create a secondary y-axis
combined_176.loc[start_date:end_date].plot(y='tmp3', ax=ax[1], title='Air and Soil Temperature', color='tab:red', x_compat=True, label='Air Temperature')
combined_176.loc[start_date:end_date].plot(y='stm010', ax=ax[1], color='tab:orange', x_compat=True, label='10cm Soil Temperature')

ax[1].axhline(0, color='tab:gray', linestyle='--', label='Freezing')
ax[1].set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
ax[1].legend(loc='upper left')

rmsp3_combined[start_date:end_date].plot(y='SWE_corr mm', ax=ax[2], title = 'Snow Water Equivalent', x_compat=True, ylabel = 'SWE (mm)', legend=False)


# Combine legends from both axes
#handles1_3, labels1_3 = ax[1].get_legend_handles_labels()  # Primary y-axis
#handles2_3, labels2_3 = secondary_y3.get_legend_handles_labels()  # Secondary y-axis
#handles_3 = handles1_3 + handles2_3
#labels_3 = labels1_3 + labels2_3
#ax[1].legend(handles_3, labels_3, loc='upper left')


for a in ax:
    a.set_xlabel(None)
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

for a in ax[list(range(0,2))]:
    a.set_xticklabels([])


fig.tight_layout()





#plt.xticks(rotation=45)


#plt.tight_layout()
#plt.show()

#ax[3].get_xticks()



In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig, ax = plt.subplots(figsize=(11, 8.5), nrows=4, sharex=False)
    ax = np.array(ax)  # Ensure ax is a numpy array for indexing

    start_date = '01/24/2025'
    end_date = '06/01/2025'

    second_y = ax[0].twinx()
    second_y.invert_yaxis()
    rme_discharge.loc[start_date:end_date].plot(y='qls', title = 'Hydrograph & Hyetograph', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    ax[0].set_ylim([.0005, 100])
    second_y.set_ylim([10, 0])

    # Combine legends from both axes
    handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    # Add the combined legend
    leg = ax[0].legend(handles, labels, loc='lower left', framealpha=.7)
    leg.set_zorder(10)
    ax[0].set_ylabel('Discharge (L/s)')
    second_y.set_ylabel('Hourly Precip (mm)')

    #secondary_y3 = ax[1].twinx()  # Create a secondary y-axis
    combined_176.loc[start_date:end_date].plot(y='tmp3', ax=ax[1], title='Air and Soil Temperature', color='tab:red', x_compat=True, label='Air Temperature')
    combined_176.loc[start_date:end_date].plot(y='stm010', ax=ax[1], color='tab:orange', x_compat=True, label='10cm Soil Temperature')

    ax[1].axhline(0, color='tab:gray', linestyle='--', label='Freezing')
    ax[1].set_ylabel('Temp (' + u'\N{DEGREE SIGN}' + 'C)')
    ax[1].legend(loc='upper left')

    soilmoisture_mbsec.loc[start_date:end_date].plot(y=['wat015', 'wat060'], title= 'Soil Moisture', x_compat=True, ylabel = 'VWC', label = ['15cm', '60cm'], ax=ax[2], color=['cyan', 'darkcyan'])

    rmsp3_combined[start_date:end_date].plot(y='SWE_corr mm', ax=ax[3], title = 'Snow Water Equivalent', x_compat=True, ylabel = 'SWE (mm)', legend=False)


    # Combine legends from both axes
    #handles1_3, labels1_3 = ax[1].get_legend_handles_labels()  # Primary y-axis
    #handles2_3, labels2_3 = secondary_y3.get_legend_handles_labels()  # Secondary y-axis
    #handles_3 = handles1_3 + handles2_3
    #labels_3 = labels1_3 + labels2_3
    #ax[1].legend(handles_3, labels_3, loc='upper left')

    ax[0].set_yticks([20, 60,100])
    ax[1].set_yticks([-20, 0, 20])
    second_y.set_yticks([0, 2 ,4, 6])
    for a in ax:
        a.set_xlabel(None)
        a.xaxis.set_major_locator(mdates.MonthLocator(bymonthday=1))
        a.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    for a in ax[list(range(0,3))]:
        a.set_xticklabels([])

    ax[3].set_xticklabels(ax[3].get_xticklabels(), rotation=0, ha='center')

    fig.tight_layout()

# Making Skinny N and Q plot

In [ ]:
pd.date_range('4/7/25', '5/12/25', freq = '2D')

In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig, ax = plt.subplots(figsize=(17,3.5), nrows = 2)
    rme_export_hourly.loc['2025-04-07':'05/12/2025'].plot(y='second_derivative_no3_mgl_bias_correct', x_compat=True, ax=ax[0], color='tab:green', legend=False, label = 'Nitrate', ylabel = 'Nitrate (mg/L)', title= 'Chemohydrograph')
    second_ax = rme_discharge.loc['2025-04-07':'05/12/2025'].plot(y='qls', ax=ax[0], color='cornflowerblue', x_compat=True, label='Discharge', legend=False, ylabel = 'Discharge (L/s)', secondary_y=True)

    soilmoisture_mbsec.loc['2025-04-07':'2025-05-12'].plot(y=['wat015', 'wat060'], title= 'Soil Moisture', legend=False, x_compat=True, ylabel = 'VWC (%)', label = ['15cm VWC', '60cm VWC'], ax=ax[1], color=['cyan', 'darkcyan'])
    rme_results_novol.loc['2025-04-07':'2025-05-12'].plot(y='Nitrate mean',label='Sample Nitrate', ax=ax[0],legend=False, marker='*', color='k',linestyle='None')


    # Get handles and labels from both axes
    handles1, labels1 = ax[0].get_legend_handles_labels()
    handles2, labels2 = second_ax.get_legend_handles_labels()
    handles3, labels3 = ax[1].get_legend_handles_labels()


    # Create a single legend in the upper right
    #ax[0].legend(handles1 + handles2, labels1 + labels2, loc='upper center', framealpha=.8)

    ax[0].set_xlabel(None)
    ax[1].set_xlabel(None)
    #second_ax.set_ylim(0, 6)
    ax[0].set_xticks(pd.date_range('1/23/25', '5/13/25', freq = '1W'))
    ax[0].set_xticklabels([])
    ax[1].set_xticks(pd.date_range('1/23/25', '5/13/25', freq = '1W'))
    ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0, ha='center')

    ax[1].set_xlim('04-07-2025', '05/12/25')
    ax[0].set_xlim('04-07-2025', '05/12/25')

    ax[1].set_yticks(np.arange(0,1, .2))
    ax[1].set_ylim(0,.4)
    ax[0].set_ylim(0, .3)
    second_ax.set_ylim(0, 200)
    fig.tight_layout()
    #ax[1].legend(loc='upper center', framealpha=.8)

    fig.legend(handles1 + handles2 + handles3, labels1 + labels2 + labels3, 
            loc='lower right',
            framealpha=1)

    # Adjust layout to make room for the legend
    fig.tight_layout()
    fig.subplots_adjust(right=0.82)  # Make space on the right


    fig.savefig('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Presentations/AGU 25/nitrate_discharge_jan_may.pdf')



# C-Q Plotting

## First ROS event

In [ ]:
def plot_CQ_discrete(df, q='qls', c='two_wavelength_no3_mgl_correct', cmap='tab20',**kwargs):
    import matplotlib.cm as cm

    # Create a new column for "period" (each 12:00 to 12:00)
    # Shift timestamps so that the "day" starts at 12:00
    shifted_index = df.index - pd.Timedelta(hours=12)
    periods = shifted_index.floor('D')
    df = df.copy()
    df['period'] = periods

    # Get unique periods and assign colors
    unique_periods = df['period'].unique()
    cmap = cm.get_cmap(cmap, len(unique_periods))
    colors = {period: cmap(i) for i, period in enumerate(unique_periods)}

    # Get the axes object from kwargs or create one
    ax = kwargs.get('ax', None)
    if ax is None:
        fig, ax = plt.subplots()

    # Plot each period as a different color
    for period in unique_periods:
        period_df = df[df['period'] == period]
        ax.scatter(
            period_df[q], period_df[c],
            color=colors[period],
            label=period.strftime('%Y-%m-%d'),
            edgecolors='k'
        )

    # Add legend
    ax.legend(title='24hr period (12:00-12:00)', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Set labels and title
    ax.set_xlabel('Discharge (L/s)')
    ax.set_ylabel('Concentration (mg/L)')
    ax.set_xscale('log')
    ax.set_yscale('log')

    scalar_formatter = ScalarFormatter()
    scalar_formatter.set_scientific(False)
    scalar_formatter.set_useOffset(False)
    ax.xaxis.set_major_formatter(scalar_formatter)
    ax.yaxis.set_major_formatter(scalar_formatter)
    ax.xaxis.set_minor_formatter(scalar_formatter)
    ax.yaxis.set_minor_formatter(scalar_formatter)
    ax.tick_params(axis='x', which='both', labelrotation=45)

    title = kwargs.get('title', 'C-Q Plot')
    ax.set_title(title)

In [ ]:
import math

def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='gray', linestyle='--')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]), fontsize=14.0)
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]), fontsize=14.0)
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2), fontsize=14.0)
    return coeff, r2

def plot_loglinear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    logdf = np.log10(data[[x,y]])
    loglineardf = np.log10(linear_range[[x,y]])
    coeff, r2 = fit_linear_function(loglineardf, x, y)
    vals= np.polyval(coeff, logdf[x])
    ax.plot(data[x], 10 ** vals, color='gray', linestyle='--')
    ax.text(text_start_x, text_start_y, 'Slope: %.2f' % (coeff[0]), fontsize=14.0)
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.2f' % (10 ** coeff[1]), fontsize=14.0)
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.2f' % (r2), fontsize=14.0)
    return coeff, r2

def plot_CQ(df, q='qls', c='second_derivative_no3_mgl_bias_correct', **kwargs):
    # Normalize index values for shading
    norm = mcolors.Normalize(vmin=df.index.min().value, vmax=df.index.max().value)
    colors = plt.cm.viridis_r(norm(df.index.values.astype(np.int64)))

    # Get the axes object from kwargs or create one
    ax = kwargs.get('ax', None)
    if ax is None:
        fig, ax = plt.subplots()

    text_start_x = kwargs.get('text_start_x', 4)
    text_start_y = kwargs.get('text_start_y', .1)
    text_y_spacing = kwargs.get('text_y_spacing', .01)

    linear_range = kwargs.get("linear_range", df)

    # Scatter plot with shading by index
    scatter = ax.scatter(df[q], df[c], c=colors, edgecolors = 'k')
    coeff, r2 = plot_loglinear_fit(df, linear_range, q, c, text_start_x, text_start_y, text_y_spacing, ax)

    # Create a ScalarMappable object for the colorbar
    sm = plt.cm.ScalarMappable(norm=norm, cmap='viridis_r')
    sm.set_array([])  # Required to avoid errors when adding the colorbar

    # Add colorbar explicitly associated with the ScalarMappable
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.invert_yaxis()

    # Format the colorbar ticks to display datetime values
    def format_datetime(value, pos):
        return pd.to_datetime(value).strftime('%b %d')

    cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(format_datetime))
   
    # Set colorbar ticks to 1-day increments at 00:00
    start = df.index.min()
    end = df.index.max()
    ticks = pd.date_range(start=start, end=end, freq='D').astype(np.int64)
    cbar.set_ticks(ticks)

    # Set labels and title
    ax.set_xlabel('Discharge (L/s)')
    ax.set_ylabel('Concentration (mg/L)')
    ax.set_xscale('log')
    ax.set_yscale('log')

    # Remove scientific notation from axis labels
    # Customize log axis formatters to avoid scientific notation
    # Remove scientific notation from axis labels
    scalar_formatter = ScalarFormatter()
    scalar_formatter.set_scientific(False)
    scalar_formatter.set_useOffset(False)

    ax.xaxis.set_major_formatter(scalar_formatter)
    ax.yaxis.set_major_formatter(scalar_formatter)
    ax.xaxis.set_minor_formatter(scalar_formatter)
    ax.yaxis.set_minor_formatter(scalar_formatter)
    ax.tick_params(axis='x', which='both', labelrotation=45)

    title=kwargs.get('title', 'C-Q Plot')

    ax.set_title(title)
    return 10 ** coeff[1], coeff[0], r2



In [ ]:
fig, ax= plt.subplots(figsize = (11,8.5))
start_date = '2/23/25 11:00'
end_date = '2/28/25 11:00'
offset, slope, r2 = plot_CQ(rme_export_hourly.loc[start_date:end_date], c='second_derivative_no3_mgl_bias_correct',cmap='viridis_r',ax=ax, title='ROS Event Rising Limb C-Q Plot', text_y_spacing = .01, text_start_x = 4, linear_range = rme_export_hourly.loc[start_date:'2025-02-24 11:00'])
ax.set_ylim(0.04, .6)
ax.set_xlim(1.5, 100)

In [ ]:
fig, ax= plt.subplots(figsize = (11,8.5))
start_date = '2/23/25 12:00'
end_date = '3/10/25 12:00'
offset, slope, r2 = plot_CQ(rme_export_hourly.loc[start_date:end_date], c='second_derivative_no3_mgl_bias_correct',cmap='viridis_r',ax=ax, title='ROS Event Falling Limb C-Q Plot', text_y_spacing = .01)
ax.set_ylim(.04, .6)
ax.set_xlim(1.5, 100)

In [ ]:
fig, ax= plt.subplots(figsize = (11,8.5))
start_date = '3/23/25 12:00'
end_date = '04/04/25 12:00'

offset, slope, r2 = plot_CQ(rme_export_hourly.loc[start_date:end_date], c='second_derivative_no3_mgl_bias_correct',cmap='viridis_r',ax=ax, title='Melt-Freeze Event C-Q Plot', text_y_spacing = .01, text_start_x=10)
ax.set_ylim(0.04, .6)
ax.set_xlim(1.5, 100)

## C-Q and Chemohydrograph

## Big ROS

In [ ]:
combined_176.loc['2/23/25 11:00':'2/28/25 11:00']['ppta_rain'].sum()

In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig = plt.figure(figsize = (14,8))
    gs0 = gridspec.GridSpec(1,2, figure = fig)
    gs00 = gridspec.GridSpecFromSubplotSpec(3,1, subplot_spec=gs0[0,0])

    hy1 = fig.add_subplot(gs00[0,0])
    temp1 = fig.add_subplot(gs00[1,0])
    sm1 = fig.add_subplot(gs00[2,0])
    cq1 = fig.add_subplot(gs0[0,1])

    start_date = '2/23/25 11:00'
    end_date = '2/28/25 11:00'

    nitrate_col = 'second_derivative_no3_mgl_bias_correct'

    second_y = temp1.twinx()
    second_y.invert_yaxis()
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    second_y.set_ylim([10, 0])



    rme_export_hourly.loc[start_date:end_date].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='Chemohydrograph', x_compat=True)
    rme_results.loc[start_date:end_date].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy1, marker='*', color='k',linestyle='None')
    #rme_results.loc[start_date:end_date].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy1, marker='^', color='k',linestyle='None')

    rme_export_hourly.loc[start_date:end_date].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)',  label='Discharge', x_compat=True)
    combined_176.loc[start_date:end_date].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True, title='Weather')
    temp1.set_ylabel('Temp (' + u'\N{DEGREE SIGN}' + 'C)')


    # Combine legends from both axes
    handles1, labels1 = temp1.get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    temp1.legend(handles, labels, loc='lower right')
    temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')

    soilmoisture_mbsec.loc[start_date:end_date].plot(y=['wat015', 'wat060'], x_compat=True, ax=sm1, label = ['15cm', '60cm'], ylabel = 'VWC (%)', color=['cyan', 'darkcyan'], title='Soil Moisture')

    plot_CQ(rme_export_hourly.loc[start_date:end_date], c=nitrate_col,cmap='viridis_r',ax=cq1, title='C-Q Plot', linear_range = rme_export_hourly.loc[start_date:'2025-02-24 11:00'])

    plot_loglinear_fit(rme_export_hourly.loc[start_date:end_date], rme_export_hourly.loc['2025-02-24 11:00':end_date], 'qls', 'second_derivative_no3_mgl_bias_correct', 1.25, .4, .05, ax=cq1)

    for ax in [hy1, temp1, sm1]:
        ax.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        ax.tick_params(axis='x', labelrotation=35)

    hy1.set_xlabel(None)
    hy1.set_xticklabels([])
    temp1.set_xticklabels([])
    sm1.set_xlabel(None)
    sm1.set_ylim(0, .4)
    cq1.set_ylim(.04, .65)
    cq1.set_xlim(1, 20)
    temp1.set_xlabel(None)
    second_y.set_ylabel('Hourly Precip (mm)')
    second_y.set_yticks([0, 4, 8])
    temp1.set_yticks([-5, 0, 5])
    sm1.set_yticks([0, .2, .4])

    fig.tight_layout()

    fig.savefig('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Presentations/AGU 25/ros.pdf')


## First Warmup

In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig = plt.figure(figsize = (14,8))
    gs0 = gridspec.GridSpec(1,2, figure = fig)
    gs00 = gridspec.GridSpecFromSubplotSpec(3,1, subplot_spec=gs0[0,0])

    hy1 = fig.add_subplot(gs00[0,0])
    temp1 = fig.add_subplot(gs00[1,0])
    sm1 = fig.add_subplot(gs00[2,0])
    cq1 = fig.add_subplot(gs0[0,1])

    start_date = '03/23/25 11:00'
    end_date = '04/04/25 11:00'

    nitrate_col = 'second_derivative_no3_mgl_bias_correct'

    second_y = temp1.twinx()
    second_y.invert_yaxis()
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    second_y.set_ylim([10, 0])

    rme_export_hourly.loc[start_date:end_date].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='Chemohydrograph', x_compat=True)
    rme_results.loc[start_date:end_date].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy1, marker='*', color='k',linestyle='None')
    #rme_results.loc[start_date:end_date].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy1, marker='^', color='k',linestyle='None')

    rme_export_hourly.loc[start_date:end_date].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)',  label='Discharge', x_compat=True)
    combined_176.loc[start_date:end_date].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True, title='Weather')
    
    # Combine legends from both axes
    handles1, labels1 = temp1.get_legend_handles_labels()  # Primary y-axis
    handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
    handles = handles1 + handles2
    labels = labels1 + labels2

    temp1.legend(handles, labels, loc= 'lower left')
    
    temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
    temp1.set_ylabel('Temp (' + u'\N{DEGREE SIGN}' + 'C)')

    soilmoisture_mbsec.loc[start_date:end_date].plot(y=['wat015', 'wat060'], x_compat=True, ax=sm1, label = ['15cm', '60cm'], ylabel = 'VWC (%)', color=['cyan', 'darkcyan'], title='Soil Moisture')

    plot_CQ(rme_export_hourly.loc[start_date:end_date], c=nitrate_col,cmap='viridis_r',ax=cq1, title='C-Q Plot', text_start_x = 7)

    #plot_loglinear_fit(rme_export_hourly.loc[start_date:end_date], rme_export_hourly.loc['2025-02-24 11:00':end_date], 'qls', 'second_derivative_no3_mgl_bias_correct', 1.25, .4, .05, ax=cq1)

    for ax in [hy1, temp1, sm1]:
        ax.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        ax.tick_params(axis='x', labelrotation=45)

    hy1.set_xlabel(None)
    hy1.set_xticklabels([])
    temp1.set_xticklabels([])
    sm1.set_xlabel(None)
    sm1.set_ylim(0, .4)
    cq1.set_ylim(.04, .65)
    cq1.set_xlim(1, 20)
    temp1.set_xlabel(None)
    second_y.set_ylabel('Hourly Precip (mm)')
    second_y.set_yticks([0, 4, 8])
    temp1.set_yticks([-10, 0, 10])
    sm1.set_yticks([0, .2, .4])

    fig.tight_layout()
    fig.savefig('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Presentations/AGU 25/warmup.pdf')


In [ ]:
fig = plt.figure(figsize = (14,8))
gs0 = gridspec.GridSpec(1,2, figure = fig)
gs00 = gridspec.GridSpecFromSubplotSpec(2,1, subplot_spec=gs0[0,0])

hy1 = fig.add_subplot(gs00[0,0])
temp1 = fig.add_subplot(gs00[1,0])
cq1 = fig.add_subplot(gs0[0,1])

start_date = '03/23/25 11:00'
end_date = '04/04/25 11:00'

nitrate_col = 'second_derivative_no3_mgl_bias_correct'

second_y = temp1.twinx()
second_y.invert_yaxis()
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
second_y.set_ylim([10, 0])

# Combine legends from both axes
handles1, labels1 = temp1.get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles = handles1 + handles2
labels = labels1 + labels2

temp1.legend(handles, labels)

rme_export_hourly.loc[start_date:end_date].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='ROS Event Chemohydrograph', x_compat=True)
rme_results.loc[start_date:end_date].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy1, marker='*', color='k',linestyle='None')
rme_results.loc[start_date:end_date].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy1, marker='^', color='k',linestyle='None')

rme_export_hourly.loc[start_date:end_date].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True)
temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
temp1.set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
plot_CQ(rme_export_hourly.loc[start_date:end_date], c=nitrate_col,cmap='viridis_r',ax=cq1, title='ROS Event C-Q Plot', text_start_x=5)

#plot_loglinear_fit(rme_export_hourly.loc[start_date:end_date], rme_export_hourly.loc['2025-02-24 11:00':end_date], 'qls', 'second_derivative_no3_mgl_bias_correct', 1.5, .4, .05, ax=cq1)

for ax in [hy1, temp1]:
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.tick_params(axis='x', labelrotation=45)

hy1.set_xlabel(None)
hy1.set_xticklabels([])
cq1.set_ylim(.04, .65)
cq1.set_xlim(1, 20)
temp1.set_xlabel(None)
second_y.set_ylabel('Hourly Precipitation (mm)')

fig.tight_layout()

In [ ]:
rme_export_daily = rme_export.drop(columns='Date').resample('1d').mean()

In [ ]:


fig = plt.figure(figsize = (14,8))
gs0 = gridspec.GridSpec(1,2, figure = fig)
gs00 = gridspec.GridSpecFromSubplotSpec(2,1, subplot_spec=gs0[0,0])

hy1 = fig.add_subplot(gs00[0,0])
temp1 = fig.add_subplot(gs00[1,0])
cq1 = fig.add_subplot(gs0[0,1])

start_date = '05/02/25 12:00'
end_date = '05/04/25 12:00'


nitrate_col = 'second_derivative_no3_mgl_bias_correct'

second_y = temp1.twinx()
second_y.invert_yaxis()
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
second_y.set_ylim([10, 0])

# Combine legends from both axes
handles1, labels1 = temp1.get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles = handles1 + handles2
labels = labels1 + labels2

temp1.legend(handles, labels)

rme_export_hourly.loc[start_date:end_date].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='ROS Event Chemohydrograph', x_compat=True)
rme_results_novol.loc[start_date:end_date].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy1, marker='*', color='k',linestyle='None')
rme_results_novol.loc[start_date:end_date].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy1, marker='^', color='k',linestyle='None')

rme_export_hourly.loc[start_date:end_date].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True)
temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
temp1.set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
plot_CQ(rme_export_hourly.loc[start_date:end_date], c=nitrate_col,cmap='viridis_r',ax=cq1, title='ROS Event C-Q Plot', text_start_x = 70, text_start_y=.19, text_y_spacing=.005)


for ax in [hy1, temp1]:
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.tick_params(axis='x', labelrotation=45)

hy1.set_xlabel(None)
hy1.set_xticklabels([])
temp1.set_xlabel(None)
second_y.set_ylabel('Hourly Precipitation (mm)')

fig.tight_layout()

### Big ROS and Warmup

In [ ]:
rme_export.columns

In [ ]:
fig = plt.figure(figsize = (11,8.5))
gs0 = gridspec.GridSpec(2,2, figure = fig)
gs00 = gridspec.GridSpecFromSubplotSpec(2,1, subplot_spec=gs0[0,0])
gs10 = gridspec.GridSpecFromSubplotSpec(2,1, subplot_spec=gs0[1,0])

hy1 = fig.add_subplot(gs00[0,0])
temp1 = fig.add_subplot(gs00[1,0])
cq1 = fig.add_subplot(gs0[0,1])

hy2 = fig.add_subplot(gs10[0,0])
temp2 = fig.add_subplot(gs10[1,0])
cq2 = fig.add_subplot(gs0[1,1])

nitrate_col = 'second_derivative_no3_mgl_bias_correct'

rme_export_hourly.loc['2/19/25 12:00':'3/02/25 12:00'].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='ROS Event Chemohydrograph', x_compat=True)
rme_results.loc['2/19/25 12:00':'3/02/25 12:00'].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy1, marker='*', color='k',linestyle='None')
rme_results.loc['2/19/25 12:00':'3/02/25 12:00'].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy1, marker='^', color='k',linestyle='None')

rme_export_hourly.loc['2/23/25 12:00':'2/28/25 12:00'].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True)
rmsp3_combined.loc['2/23/25 12:00':'2/28/25 12:00'].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True)
temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
temp1.set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
plot_CQ_discrete(rme_export_hourly.loc['2/23/25 12:00':'2/28/25 12:00'], c=nitrate_col,cmap='viridis_r',ax=cq1, title='ROS Event C-Q Plot')

rme_export_hourly.loc['3/23/25 12:00':'4/4/25 12:00'].plot(y=nitrate_col, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', title='Melt/Freeze Event Chemohydrograph', ax=hy2, x_compat=True)
rme_results_novol.loc['3/23/25 12:00':'4/4/25 12:00'].plot(y='Nitrate mean',label='Sample Nitrate', ax=hy2, marker='*', color='k',linestyle='None')
rme_results_novol.loc['3/23/25 12:00':'4/4/25 12:00'].plot(y='Ammonium mean',label='Sample Ammonium', ax=hy2, marker='^', color='k',linestyle='None')

rme_export_hourly.loc['3/23/25 12:00':'4/4/25 12:00'].plot(y='qls', ax=hy2, secondary_y=True, color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True)
plot_CQ_discrete(rme_export_hourly.loc['3/23/25 12:00':'4/4/25 12:00'], c=nitrate_col, cmap='viridis_r',ax=cq2, title='Melt/Freeze Event C-Q Plot', x_compat=True)
rmsp3_combined.loc['3/23/25 12:00':'4/4/25 12:00'].plot(y='tmp3', ax=temp2, color='tab:red', label='Air Temperature', x_compat=True)
temp2.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
temp2.set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')

for ax in [hy1, temp1, hy2, temp2]:
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.tick_params(axis='x', labelrotation=45)

hy1.set_xlabel(None)
hy1.set_xticklabels([])
temp1.set_xlabel(None)
hy2.set_xlabel(None)
hy2.set_xticklabels([])
temp2.set_xlabel(None)

fig.tight_layout()

### Daily Warmups

In [ ]:
fig, ax = plt.subplots(nrows= 5, figsize=(11,8.5), sharey=True, layout='constrained')

nitrate_col = 'second_derivative_no3_mgl_bias_correct'

rme_export_hourly.loc['2025-04-07':'2025-04-13'].plot(y=nitrate_col, ax=ax[0], label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-07':'2025-04-13'].plot(y='qls', secondary_y=True, ax=ax[0], color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-14':'2025-04-20'].plot(y=nitrate_col, ax=ax[1], label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-14':'2025-04-20'].plot(y='qls', secondary_y=True, ax=ax[1], color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-21':'2025-04-27'].plot(y=nitrate_col, ax=ax[2], label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-21':'2025-04-27'].plot(y='qls', secondary_y=True, ax=ax[2], color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-28':'2025-05-04'].plot(y=nitrate_col, ax=ax[3], label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True, legend=False)
rme_export_hourly.loc['2025-04-28':'2025-05-04'].plot(y='qls', secondary_y=True, ax=ax[3], color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True, legend=False)
rme_export_hourly.loc['2025-05-05':'2025-05-12'].plot(y=nitrate_col, ax=ax[4], label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True, legend=False)
rme_export_hourly.loc['2025-05-05':'2025-05-12'].plot(y='qls', secondary_y=True, ax=ax[4], color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True, legend=False)

for a in ax:
    a.set_xlabel(None)
    a.xaxis.set_major_locator(mdates.HourLocator(byhour=12))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

handles = ax[0].get_lines() + ax[0].right_ax.get_lines()
labels = [line.get_label() for line in handles]

#fig.legend(handles, labels, loc='outside right lower')
fig.suptitle('Melt Freeze Chemohydrographs')
fig.tight_layout()

- flow is confusingly at a minimum at noon
- hysteresis pattern shifts over the course of the melt out - starts as counterclockwise, like big warmup, but then transitions to diluting- lowest nitrate at highest discharge

In [ ]:
def plot_event_figure(cq_df, wx_df, start_date, end_date, c_range, q_range, t_range, sm_df, sm_range, nitrate_col='second_derivative_no3_mgl_bias_correct', **kwargs):
    fig = plt.figure(figsize = (13, 5.5))
    gs0 = gridspec.GridSpec(1,2, figure = fig)
    gs00 = gridspec.GridSpecFromSubplotSpec(3,1, subplot_spec=gs0[0])
    hy1 = fig.add_subplot(gs00[0,0])
    temp1 = fig.add_subplot(gs00[1,0])
    sm1 = fig.add_subplot(gs00[2,0])
    cq1 = fig.add_subplot(gs0[1])


    second_y = temp1.twinx()
    second_y.invert_yaxis()
    second_y.bar(wx_df.loc[start_date:end_date].index,wx_df.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
    second_y.bar(wx_df.loc[start_date:end_date].index,wx_df.loc[start_date:end_date]['ppta_snow'],width = pd.Timedelta(1,'hour'), color='tab:orange', label = 'Snow')
    second_y.set_ylim([10, 0])

    cq_df.loc[start_date:end_date].plot(y=nitrate_col, ax=hy1, label = 'Nitrate', color='tab:green', ylabel='Concentration (mg/L)', x_compat=True)
    secondy = cq_df.loc[start_date:end_date].plot(y='qls', secondary_y=True, ax=hy1, color = 'cornflowerblue', ylabel = 'Discharge (L/s)', label='Discharge', x_compat=True)
    wx_df.loc[start_date:end_date].plot(y='tmp3', ax=temp1, color='tab:red', label='Air Temperature', x_compat=True)
    temp1.axhline(0, color='tab:gray', linestyle='--', label='Freezing')
    temp1.set_ylabel('Temperature (' + u'\N{DEGREE SIGN}' + 'C)')
    intercept, slope, r2 = plot_CQ(cq_df.loc[start_date:end_date], ax=cq1, **kwargs)
    hy1.set_xlabel(None)
    hy1.set_xticklabels([])
    temp1.set_xlabel(None)
    temp1.set_xticklabels([])
    sm1.set_xlabel(None)

    sm_df.loc[start_date:end_date].plot(y=['wat015', 'wat060'], x_compat=True, ax=sm1, label = ['15cm', '60cm'], ylabel = 'Volumetric Water Content', color=['cyan', 'darkcyan'], title='Soil Moisture')


    temp1.set_ylim(t_range)
    cq1.set_ylim(c_range)
    sm1.set_ylim(sm_range)
    cq1.set_xlim(q_range)
    hy1.set_ylim(c_range)
    cq1.set_title(None)
    secondy.set_ylim(q_range)


    fig.suptitle(start_date + ' - ' + end_date)
    fig.tight_layout()

    return fig, intercept, slope, r2

    



In [ ]:
def plot_event_figures(cq_df, wx_df, start_dates, end_dates, c_range, q_range, t_range, sm_df, sm_range, filetype = '.pdf', **kwargs):
    result_df = pd.DataFrame(columns = ['Intercept', 'Slope', 'R2'])
    for start_date, end_date in zip(start_dates, end_dates):
        fig, intercept, slope, r2 = plot_event_figure(cq_df, wx_df, start_date, end_date, c_range, q_range, t_range, sm_df, sm_range, **kwargs)
        result_df.loc[pd.to_datetime(start_date)] = (intercept, slope, r2)
        output_path = kwargs.get('output_path')
        
        if output_path is not None:
            fig.savefig(output_path + start_date.replace('/', '_').replace(':', '') +filetype)

    return result_df

In [ ]:
def generate_daily_dates(start_date, end_date):
    """
    Generate a list of date strings formatted as M/D/Y 12:00, spaced one day apart.

    Args:
        start_date (str): The start date in the format 'M/D/Y'.
        end_date (str): The end date in the format 'M/D/Y'.

    Returns:
        list: A list of date strings formatted as 'M/D/Y 12:00'.
    """
    # Parse the start and end dates
    start = datetime.strptime(start_date, '%m/%d/%Y')
    end = datetime.strptime(end_date, '%m/%d/%Y')
    
    # Generate the list of dates
    dates = [(start + timedelta(days=i)).strftime('%m/%d/%Y 11:00') 
             for i in range((end - start).days + 1)]
    
    return dates


In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    output_path = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Presentations/New CQ Plots/'

    melt_results = plot_event_figures(rme_export, rmsp3_combined, generate_daily_dates('4/07/2025', '05/11/2025'), generate_daily_dates('4/08/2025', '5/12/2025'), (.05, .35), (5,100), (-15, 20), soilmoisture_mbsec, (0, .4), filetype = '.pdf', output_path=output_path, text_start_y = .28, text_y_spacing = .025, text_start_x = 6)

In [ ]:
with plt.rc_context({'axes.titlesize': 18, 
                     'axes.labelsize': 16,
                     'xtick.labelsize': 14,
                     'legend.fontsize': 14,
                     'ytick.labelsize': 14}):

    fig, ax = plt.subplots(figsize = (6,6))
    melt_results.loc[:'5/11/2025'].drop([pd.to_datetime('4/17/2025 11:00'), pd.to_datetime('4/18/2025 11:00'), pd.to_datetime('4/19/2025 11:00')]).plot(y='Slope', ax=ax, label = 'Exponent', title = 'Diurnal Melt Cycle C-Q Exponent vs. Time', marker='o', legend=False, x_compat=True)

    ax.axhline(0.15, linestyle = '--', color='gray')
    ax.axhline(-0.15, linestyle = '--', color='gray')
    ax.set_ylim(-1.2, 1.2)
    #ax.set_yticks([-1, -.8, -.6, -.4, -.2, 0, .2, .4, .6, .8, 1])
    ax.axhspan(-0.15, 0.15, color='yellow', alpha=0.3)
    ax.axhspan(.15, 1.2, color='lightcoral', alpha=0.1)
    ax.axhspan(-1.2, -0.15, color='skyblue', alpha=0.1)
    ax.text(pd.to_datetime('2025-04-28 11:00'), 1, "Concentrating", fontsize=14)
    ax.text(pd.to_datetime('2025-04-28 11:00'), 0, "Near Chemostatic", fontsize=14)
    ax.text(pd.to_datetime('2025-04-28 11:00'), -1, "Diluting", fontsize=14)
    ax.set_xticks(pd.date_range('4/07/25', '5/13/25', freq = '1W'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.tick_params(axis='x', labelrotation=35)
    ax.set_title('Diurnal Melt Cycle C-Q Slope vs. Time', fontsize=14)





In [ ]:
combined_176.loc['04/27/2025 11:00':'04/28/2025 11:00']['ppta_rain'].sum()

In [ ]:
combined_176.loc['04/25/2025 11:00':'04/26/2025 11:00']['ppta_rain'].sum()

In [ ]:
combined_176.loc['05/03/2025 11:00':'05/04/2025 11:00']['ppta_rain'].sum()